# Delta Live Tables — Complete Cheat Sheet


---

## Core Concepts

| Term | Meaning |
|---|---|
| **Delta Table** | Storage format (ACID, time travel) |
| **DLT** | Declarative pipeline framework that manages Delta tables |
| **Streaming Table** | Processes only new data (incremental, append-only) |
| **Materialized View** | Recomputes all data on refresh (aggregations, joins) |
| **View** | Saved query, no data stored, not published |
| **"Live"** | Means "actively managed by the pipeline", **not** real-time |

## The 3 Dataset Types — When to Use

| Layer | Type | Reason |
|---|---|---|
| Bronze (raw) | Streaming Table | Incremental ingestion |
| Silver (clean) | Streaming Table or MV | Row-level transforms = ST; joins/updates = MV |
| Gold (aggregate) | Materialized View | `GROUP BY`, `SUM`, `COUNT` |
| Intermediate | View or `TEMPORARY` | Doesn't need publishing |

> **Rule:** Streaming Tables feed anything. Only append-only sources feed Streaming Tables.

## Addressing Rules (CRITICAL)

| Where you run | How to reference |
|---|---|
| Notebook | `workspace.default.table_name` |
| Pipeline | `live.table_name` |
| Streaming read | `FROM STREAM(live.x)` |
| Batch read | `FROM live.x` |

> Mixing them = confusing errors. Pick one.

## Expectations (Data Quality)

### Three Actions

| SQL Action | Python Decorator | UI Name |
|---|---|---|
| (default) or `WARN` | `@dlt.expect` | ALLOW |
| `ON VIOLATION DROP ROW` | `@dlt.expect_or_drop` | DROP |
| `ON VIOLATION FAIL UPDATE` | `@dlt.expect_or_fail` | FAIL |

### SQL Syntax

```sql
CONSTRAINT rule_name EXPECT (condition) ON VIOLATION <action>
```

### Python Syntax

```python
@dlt.expect("rule_name", "condition")              # WARN
@dlt.expect_or_drop("rule_name", "condition")      # DROP
@dlt.expect_or_fail("rule_name", "condition")      # FAIL
```

### Multiple Expectations at Once (Python)

```python
rules = {"rule1": "cond1", "rule2": "cond2"}
@dlt.expect_all(rules)              # all WARN
@dlt.expect_all_or_drop(rules)      # all DROP
@dlt.expect_all_or_fail(rules)      # all FAIL
```

### Key Rules

- `EXPECT` runs on **source columns** (before renaming)
- Order matters: evaluated top-to-bottom
- Put **FAIL** rules first, **WARN** rules last
- WARN is the default, so no `ON VIOLATION` is needed

## SQL vs Python Quick Reference

| SQL | Python |
|---|---|
| `CREATE OR REFRESH STREAMING TABLE x` | `@dlt.table(name="x")` |
| `CREATE OR REFRESH MATERIALIZED VIEW x` | `@dlt.table(name="x")` (no STREAM read) |
| `CONSTRAINT n EXPECT (c) DROP ROW` | `@dlt.expect_or_drop("n", "c")` |
| `FROM STREAM(live.x)` | `dlt.read_stream("x")` |
| `FROM live.x` | `dlt.read("x")` |
| `read_files(...)` | `spark.readStream.format("cloudFiles")` |
| `${param}` | `spark.conf.get("param")` |

## CDC (APPLY CHANGES)

### SQL

```sql
CREATE OR REFRESH STREAMING TABLE target;

APPLY CHANGES INTO live.target
FROM STREAM(live.source)
KEYS (id)
SEQUENCE BY sequenceNum
APPLY AS DELETE WHEN operation = 'DELETE'
COLUMNS * EXCEPT (operation, sequenceNum)
STORED AS SCD TYPE 1;   -- or TYPE 2 for history
```

### Python

```python
dlt.apply_changes(
    target="target",
    source="source",
    keys=["id"],
    sequence_by=col("sequenceNum"),
    apply_as_deletes=expr("operation = 'DELETE'"),
    except_column_list=["operation", "sequenceNum"],
    stored_as_scd_type=1
)
```

### Key Rules

- Target must be a **Streaming Table**
- `SEQUENCE BY` column must be sortable
- **SCD Type 1** = current state only
- **SCD Type 2** = full history (`__START_AT`, `__END_AT`)
- Downstream reads must use a **Materialized View**

## Parameters

Define in **Pipeline Settings → Configuration**.

**SQL:** `${param_name}`

```sql
FROM read_files('${source_path}')
WHERE clicks > CAST('${min_clicks}' AS INT)
```

**Python:** `spark.conf.get("param_name")`

```python
path = spark.conf.get("source_path")
```

### Rules

- All params are strings, so cast when needed
- A change requires a pipeline restart
- Missing param = error (define defaults)

## Product Editions

| Edition | Unlocks | Cost |
|---|---|---|
| **Core** | Streaming ingest only | 💰 |
| **Pro** | Core + CDC (`APPLY CHANGES`) | 💰💰 |
| **Advanced** | Pro + Expectations | 💰💰💰 |

> **Rule:** Expectations require Advanced. CDC requires Pro or Advanced.

## Update Modes

| Mode | Stops When? | Best For |
|---|---|---|
| **Triggered** | Automatically when done | Daily/hourly runs |
| **Continuous** | Only when you stop it | Real-time (10s–minutes) |

## Refresh Types

| Type | Behavior |
|---|---|
| **Refresh all** | Appends new data (ST), recomputes (MV) |
| **Full refresh all** | Drops & rebuilds everything |
| **Selective refresh** | Only works on Triggered pipelines |

> Use **Full refresh** when the schema changes (fixes `DELTA_METADATA_MISMATCH`).

## Development vs Production Modes

| | Development | Production |
|---|---|---|
| Cluster | Reused | Restarts on errors |
| Retries |  Off |  On |
| Idle shutdown | 2 hours | 0 (always on) |
| Best for | Testing | Live systems |

> Switching modes doesn't change storage, only runtime behavior.

## Publishing Data

| | Hive Metastore (Legacy) | Unity Catalog (Modern) |
|---|---|---|
| Scope | Per-workspace | Account-wide |
| Namespace | 2-level | 3-level (`catalog.schema.table`) |
| Security | Basic ACLs | Row/column-level |
| Lineage |  |  Automatic(Unity Catalog provides automatic, runtime data lineage. It tracks both table-level lineage (which source tables feed into a target table) and column-level lineage (which specific source columns were used to create target columns, including derived columns). It also links the Databricks Jobs and notebooks that executed the query. This is incredibly useful for impact analysis—for example, if I need to rename a column in the Bronze layer, I can look at the lineage graph to see exactly which Silver and Gold columns and dashboards will break.) |
| Migration |  | Must rebuild pipeline |

### Rules

- No target set → tables stay private to the pipeline
- Delete pipeline → **all its tables get deleted** ⚠️
- Use `TEMPORARY` for hidden intermediate tables
- One pipeline can't write to both Hive and Unity Catalog

## Scheduling

- **Schedule** button in pipeline UI → creates a Job
- Two trigger types: **Simple** (intervals) vs **Advanced** (cron)
- **Queue enabled** → prevents overlapping runs

## Alerts

- **Pipeline Settings → Notifications**
- Triggers: success, failure, data flow failure
- Advanced: **Event Hooks** for Slack/custom callbacks

## Event Log

### Where it lives

| Catalog | Location |
|---|---|
| Hive | `/pipelines/<id>/system/events` (Delta table) |
| Unity Catalog | Accessed via `event_log()` function |

### Query via Unity Catalog

```sql
SELECT * FROM event_log("<pipeline-id>");
-- or
SELECT * FROM event_log(TABLE(catalog.schema.table));
```

### Common event types

| Event | Contains |
|---|---|
| `flow_definition` | Lineage (input/output datasets) |
| `flow_progress` | Data quality, backlog |
| `autoscale` | Cluster resizing |
| `cluster_resources` | Compute utilization |
| `user_action` | Audit logs |
| `create_update` | Runtime version, config |

> **Requirements:** Shared cluster or SQL Warehouse + pipeline owner.

## Common Errors & Fixes

| Error | Cause | Fix |
|---|---|---|
| `UNRESOLVED_COLUMN` | Wrong column name in EXPECT | Use source column names, not aliases |
| `DELTA_METADATA_MISMATCH` | Schema changed | Full refresh all |
| `RESOURCE_EXHAUSTED` | Daily quota hit | Wait for reset |
| `ST_NOT_ENABLED_ON_SERVERLESS` | Compute type incompatible | Switch to SQL Warehouse |
| Syntax error at or near `'WARN'` | `WARN` isn't valid SQL | Remove `ON VIOLATION WARN` (it's the default) |

## Golden Rules

1. One pipeline, many tables: keep all logic in one file
2. Rename once, in the raw layer: don't rename twice
3. `live.` inside the pipeline, `workspace.default.` outside
4. Pause the schedule while developing
5. `LIMIT 10` for Free Edition testing
6. Full refresh when the schema changes
7. `EXPECT` needs source column names
8. Order matters: FAIL rules first, WARN rules last

## ACID Properties

| Letter | Meaning |
|---|---|
| **A** | Atomicity: all or nothing |
| **C** | Consistency: rules always hold |
| **I** | Isolation: no interference |
| **D** | Durability: once saved, always saved |

> Delta Lake brings ACID to data lakes via the transaction log (`_delta_log/`).

## Quick Comparison: Jobs vs DLT Pipelines

| Feature | DLT Pipeline | Databricks Job |
|---|---|---|
| Purpose | Manage tables & data flow | Orchestrate any task |
| Unit | Datasets (tables/views) | Tasks (notebooks) |
| Dependencies | Automatic (from `live.`) | Manual (`depends_on`) |
| Loops | ❌ | ✅ `for_each_task` |
| Best for | ETL/data engineering | Complex orchestration |

> Real projects often use **both**.

## Your Learning Path Completed

- ✅ DLT basics
- ✅ Streaming Tables vs Materialized Views vs Views
- ✅ Expectations (WARN, DROP, FAIL)
- ✅ Multiple Expectations
- ✅ CDC (APPLY CHANGES)
- ✅ SQL vs Python syntax
- ✅ Parameters
- ✅ Product Editions (Core/Pro/Advanced)
- ✅ Dev vs Prod modes
- ✅ Hive vs Unity Catalog
- ✅ Scheduling and Alerts
- ✅ Event Log and Monitoring
- ✅ ACID properties
- ✅ Jobs vs DLT Pipelines

In [0]:
%sh
pwd

In [0]:
%sh
# Create a working directory on the driver
mkdir -p /tmp/practice
cd /tmp/practice

# Download the file
curl -sSL -o titanic.csv https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

# Verify it downloaded
ls -lh titanic.csv
echo "---"
echo "Line count:"
wc -l titanic.csv
echo "---"
echo "First 5 rows:"
head -5 titanic.csv